# One pass per question: Signpost with every option in one input

**Part 4b of "LLMs the Hard Way"** (branch `signpost`), next to `03_lora_signpost.ipynb`. Signpost reads the conversation once **per option**; here every option goes into **one** input, the encoder runs **once**, and one shared head scores each option at its own tokens.

```text
<history> [SEP] current agent: X [SEP] question: Q [SEP] option: Billing Agent. Handles… [SEP] option: Returns Agent. … [SEP] …
   → ModernBERT-large + LoRA (one pass) → the mean of each option's tokens → the same head on each → softmax over the N options
```

**Recipe** (run 28; the others are in *What we tried and dropped*): `MAX_LEN` 1,536; Signpost's data plus 2,500 many-option questions; distractor agents for routing; plain cross-entropy; batches by token budget; LoRA through `peft`.

**Tests:** every set of `03`, fresh many-option and many-agent sets, and an option-order check; Signpost's and the decider's numbers are copied, not rerun. Runs on Kaggle with a T4 and Internet on.

In [1]:
!pip uninstall -y -q torchao      # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import copy, gc, json, os, random, shutil, statistics, subprocess, sys, time
from collections import Counter
from pathlib import Path

import peft
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from peft import LoraConfig, get_peft_model
from transformers import AutoModel, AutoTokenizer

# Settings: each can be changed with an environment variable of the same name
SMOKE = os.environ.get("LORA_SMOKE") == "1"             # 1 = a tiny code check on ModernBERT-base
ENCODER = "answerdotai/ModernBERT-base" if SMOKE else "answerdotai/ModernBERT-large"
LORA_R = int(os.environ.get("LORA_R", "16"))
LORA_ALPHA = float(os.environ.get("LORA_ALPHA", "32"))
LORA_DROPOUT = 0.1
LR = float(os.environ.get("LR", "2e-4"))                # the LoRA adapters
HEAD_LR = float(os.environ.get("HEAD_LR", "1e-3"))      # the scoring head
EPOCHS = int(os.environ.get("EPOCHS", "1" if SMOKE else "2"))
MAX_LEN = int(os.environ.get("MAX_LEN", "1536"))        # tokens: history + question + all options
MAX_OPTIONS = int(os.environ.get("MAX_OPTIONS", "50"))  # more options raise an error
AUG_MAX_AGENTS = int(os.environ.get("AUG_MAX_AGENTS", "15"))          # most agents in training
MANY_AGENT_ROUTES = int(os.environ.get("MANY_AGENT_ROUTES", "2500"))  # routing items with 10-15
TOKEN_BUDGET = int(os.environ.get("TOKEN_BUDGET", "8000"))  # tokens a batch, padding included
MAX_BATCH = int(os.environ.get("MAX_BATCH", "16"))          # questions a batch at most
SEED = int(os.environ.get("SEED", "42"))                # the first seed; +1000 per retry
SAVE_MODEL = os.environ.get("SAVE_MODEL", "1") == "1"   # save, zip and reload-check the model
SEP = " [SEP] "
ROUTE_Q = "Which agent should handle the user's latest message?"

The device, a progress log (also written to `one_pass_run.log`), and the data folder: this repository's, or a clone of the `signpost` branch.

In [2]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"                    # fp16 mixed precision on the GPU
RUN_LOG = Path("one_pass_run.log")

def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")

def gpu():
    if not AMP:
        return "CPU"
    used = torch.cuda.memory_allocated(DEVICE) / 1e9
    peak = torch.cuda.max_memory_allocated(DEVICE) / 1e9
    return f"GPU {used:.1f} GB (peak {peak:.1f})"

versions = (f"torch {torch.__version__} | transformers {transformers.__version__} | "
            f"peft {peft.__version__}")
gpu_name = f" ({torch.cuda.get_device_name(DEVICE)})" if AMP else ""
log(f"{versions} | {DEVICE}{gpu_name} | {ENCODER}" + (" | SMOKE CHECK" if SMOKE else ""))

DATA_BRANCH = "signpost"
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():                     # on Kaggle / Colab: clone this branch, or refresh it
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        git = ["git", "-C", str(repo)]
        subprocess.run(git + ["fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(git + ["checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data" / "modernbert_router"

def load(path):
    return json.load(open(DATA_DIR / path))

[06:54:37] torch 2.11.0+cu130 | transformers 5.17.0 | peft 0.21.2 | cuda:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition) | answerdotai/ModernBERT-large


Routing and question items in one shape (messages, current agent, question, options, answer), and the dev split of `03`: 10% of businesses, 10% of texts.

In [3]:
domains = {**load("generic/domains.json"), **load("realistic_train/domains.json")}

def route_item(ex):
    return {"task": "routing", "domain": ex["domain"], "messages": ex["messages"],
            "current": ex["current_agent"], "question": ROUTE_Q,
            "options": domains[ex["domain"]]["agents"], "answer": ex["target_agent"]}

def choice_item(it):
    return {"task": it["type"], "domain": it["domain"],
            "messages": [{"role": "user", "content": it["state"]}], "current": None,
            "question": it["question"], "options": {o: "" for o in it["options"]},
            "answer": it["answer"], "state": it["state"]}

route_examples = load("generic/conversations.json") + load("realistic_train/conversations.json")
route_items = [route_item(ex) for ex in route_examples]
choice_examples = (load("choice_train/items.json")["items"]
                   + load("many_train/items.json")["items"])            # + 2,500 many-option
choice_items = [choice_item(it) for it in choice_examples]

rng = random.Random(SEED)
route_domains = sorted({x["domain"] for x in route_items})
rng.shuffle(route_domains)
dev_domains = set(route_domains[: max(1, len(route_domains) // 10)])
texts = sorted({x["state"] for x in choice_items})
rng.shuffle(texts)
dev_texts = set(texts[: len(texts) // 10])
train_route = [x for x in route_items if x["domain"] not in dev_domains]
dev_route = [x for x in route_items if x["domain"] in dev_domains]
train_choice = [x for x in choice_items if x["state"] not in dev_texts]
dev_choice = [x for x in choice_items if x["state"] in dev_texts]

Distractor agents come from other businesses and never share a name word with an item's own agents. 2,500 routing items are marked for 10–15 agents; dev routing gets a fixed 15.

In [4]:
GENERIC = {"general", "support", "help", "customer", "customers", "billing", "payment", "payments",
           "account", "accounts", "booking", "bookings", "enquiries", "service", "services", "care",
           "desk", "team"}

def name_words(name):
    return {w.strip(",.&'").lower() for w in name.split()} - {"agent", "and", "&", ""}

DISTRACTORS = [(n, d, dom) for dom, info in domains.items() for n, d in info["agents"].items()
               if not name_words(n) & GENERIC]

def add_distractors(item, rng, n_total):
    """The item plus other businesses' agents, up to n_total agents in all."""
    own = set().union(*(name_words(n) for n in item["options"]))
    pool = [(n, d) for n, d, dom in DISTRACTORS
            if dom != item["domain"] and not name_words(n) & own]
    n_extra = max(0, min(len(pool), n_total - len(item["options"])))
    return {**item, "options": {**item["options"], **dict(rng.sample(pool, n_extra))}}

for x in random.Random(SEED).sample(train_route, min(MANY_AGENT_ROUTES, len(train_route))):
    x["many_agents"] = True              # trained with 10-15 agents each time; the others with 5-9
dev_route_many = [add_distractors(x, random.Random(SEED + k), AUG_MAX_AGENTS)
                  for k, x in enumerate(dev_route)]
if SMOKE:
    train_route, dev_route = train_route[::400], dev_route[::100]
    train_choice, dev_choice = train_choice[::400], dev_choice[::100]
    dev_route_many = dev_route_many[::100]

for name, xs in [("train routing", train_route), ("train questions", train_choice),
                 ("dev routing", dev_route), ("dev questions", dev_choice)]:
    print(f"{name:15s}: {len(xs):5d} | {dict(Counter(x['task'] for x in xs))}")
n_many_agents = sum(x.get("many_agents", False) for x in train_route)
n_many_options = sum(x["domain"].startswith("many/") for x in train_choice)
print(f"routing items trained with 10-15 agents: {n_many_agents} | "
      f"many-option questions: {n_many_options} | distractor agents: {len(DISTRACTORS)}")

train routing  :  6319 | {'routing': 6319}
train questions: 11730 | {'noul': 5000, 'choice': 4788, 'score': 1942}
dev routing    :   531 | {'routing': 531}
dev questions  :  1370 | {'noul': 620, 'choice': 544, 'score': 206}
routing items trained with 10-15 agents: 2500 | many-option questions: 2279 | distractor agents: 309


The test sets of `03`, plus the fresh many-option and many-agent sets; E10 is also counted at its 6 test companies.

In [5]:
real_convs = load("realistic/conversations.json")                                 # E10
ft = load("first_turn/messages.json")
e11 = [{**m, "agents": ft["companies"][m["company"]]} for m in ft["messages"]]     # E11
banking = load("banking/benchmark.json") + load("banking/extended.json")
e12 = load("choice_test/items.json")["items"]
ind_q = load("independent/questions.json")["items"]
ind_r = load("independent/routing.json")
ind_f = load("independent/first_turn.json")["messages"]
ind2_q = load("independent2/questions.json")["items"]
ind2_r = load("independent2/routing.json")
ind2_f = load("independent2/first_turn.json")["messages"]
many_q = load("many_test/questions.json")["items"]
many_r = load("many_test/routing.json")
many_f = load("many_test/first_turn.json")["messages"]
companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])   # the 6 test companies of 02 and 03
if SMOKE:
    real_convs, e11, banking, e12 = real_convs[:2], e11[::20], banking[:6], e12[::20]
    ind_q, ind_r, ind_f = ind_q[::30], ind_r[:2], ind_f[::20]
    ind2_q, ind2_r, ind2_f = ind2_q[::20], ind2_r[:2], ind2_f[::10]
    many_q, many_r, many_f = many_q[::20], many_r[:2], many_f[::12]

def turns(convs):
    return sum(len(c["user_turns"]) for c in convs)

print(f"E10 {turns(real_convs)} turns | E11 {len(e11)} | banking {len(banking)} | E12 {len(e12)} | "
      f"independent {len(ind_q)} questions, {turns(ind_r)} turns, {len(ind_f)} first | "
      f"independent 2 {len(ind2_q)} questions, {turns(ind2_r)} turns, {len(ind2_f)} first")
option_counts = [len(q["options"]) for q in many_q]
agent_counts = [m["n_agents"] for m in many_f]
print(f"many-option: {len(many_q)} questions ({min(option_counts)}-{max(option_counts)} options) | "
      f"many-agent: {turns(many_r)} turns, {len(many_f)} first messages "
      f"({min(agent_counts)}-{max(agent_counts)} agents)")

E10 193 turns | E11 100 | banking 225 | E12 170 | independent 300 questions, 172 turns, 200 first | independent 2 200 questions, 70 turns, 50 first
many-option: 162 questions (10-50 options) | many-agent: 120 turns, 72 first messages (9-15 agents)


One input per question, options last with their character spans; the oldest messages are dropped until it fits `MAX_LEN`. The mask marks which tokens belong to each option.

In [6]:
def build_input(item, drop=0):
    """-> (text, character span of each option), with the first `drop` messages left out."""
    parts = [f"{m['role']}: {m['content']}" for m in item["messages"][drop:]]
    parts += [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}"]
    text = SEP.join(parts)
    spans = []
    for name, desc in item["options"].items():
        seg = f"{name}. {desc}" if desc else name
        text += SEP + "option: "
        spans.append((len(text), len(text) + len(seg)))
        text += seg
    return text, spans

def fit_input(tok, item):
    """Drop the oldest messages until the input fits MAX_LEN; the options always stay."""
    n_options = len(item["options"])
    assert n_options <= MAX_OPTIONS, f"{n_options} options > MAX_OPTIONS={MAX_OPTIONS}"
    for drop in range(len(item["messages"])):
        text, spans = build_input(item, drop)
        if len(tok(text)["input_ids"]) <= MAX_LEN:
            return text, spans
    raise ValueError(f"{n_options} options don't fit in MAX_LEN={MAX_LEN}, "
                     "even with only the latest message")

def option_token_mask(all_spans, attention_mask, offsets):
    """-> [B, K, T] mask on DEVICE: 1 where token t lies in option k of question b."""
    K = max(len(sp) for sp in all_spans)
    spans = torch.tensor([sp + [(0, 0)] * (K - len(sp)) for sp in all_spans])   # [B, K, 2]
    s, e = offsets[..., 0], offsets[..., 1]                                     # [B, T]
    real = attention_mask.bool() & (e > s)                                      # real tokens only
    starts_before_end = s[:, None] < spans[..., 1:]
    ends_after_start = e[:, None] > spans[..., :1]
    return (real[:, None] & starts_before_end & ends_after_start).float().to(DEVICE)

ModernBERT with LoRA and a small head. `scores` runs one encoder pass, averages each option's tokens and scores them with the same head; missing options get −10,000.

In [7]:
class OnePassModel:
    def __init__(self):
        self.tok = AutoTokenizer.from_pretrained(ENCODER)
        self.tok.padding_side = "right"
        enc = AutoModel.from_pretrained(ENCODER, dtype=torch.float32).to(DEVICE)
        self.H = enc.config.hidden_size
        lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                          target_modules=["Wqkv", "Wo", "Wi"])
        self.enc = get_peft_model(enc, lora)
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(self.H, 256), nn.GELU(),
                                  nn.Dropout(0.1), nn.Linear(256, 1)).to(DEVICE)

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode)
        self.head.train(mode)

    def scores(self, items):
        """-> [items, most options] scores, missing options -1e4."""
        built = [fit_input(self.tok, it) for it in items]
        enc = self.tok([t for t, _ in built], padding=True, return_offsets_mapping=True,
                       return_tensors="pt")
        offsets = enc.pop("offset_mapping")                         # [B, T, 2]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            out = self.enc(input_ids=enc["input_ids"].to(DEVICE),
                           attention_mask=enc["attention_mask"].to(DEVICE))
            hidden = out.last_hidden_state                          # [B, T, H]
        all_spans = [sp for _, sp in built]
        member = option_token_mask(all_spans, enc["attention_mask"], offsets)   # [B, K, T]
        counts = member.sum(-1)                                                 # [B, K]
        n_options = sum(len(sp) for sp in all_spans)
        assert int((counts > 0).sum()) == n_options, "an option got no tokens"
        pooled = (member @ hidden.float()) / counts.clamp(min=1).unsqueeze(-1)  # [B, K, H] mean
        return self.head(pooled).squeeze(-1).masked_fill(counts == 0, -1e4)

    def probs(self, item):
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()

Length-sorted batches within `TOKEN_BUDGET` tokens and `MAX_BATCH` items, the optimizer with warm-up and linear decay, and fresh distractors and shuffled options for every batch.

In [8]:
def est_tokens(it):
    chars = sum(len(m["content"]) + 12 for m in it["messages"]) + len(it["question"]) + 40
    chars += sum(len(n) + len(d) + 12 for n, d in it["options"].items())
    if it["task"] == "routing":
        chars += max(0, AUG_MAX_AGENTS - len(it["options"])) * 130
    return min(MAX_LEN, chars // 4 + 8)

def make_batches(items):
    batches, cur, longest = [], [], 0
    for it in sorted(items, key=est_tokens):
        n = est_tokens(it)
        too_many_tokens = max(longest, n) * (len(cur) + 1) > TOKEN_BUDGET
        if cur and (too_many_tokens or len(cur) >= MAX_BATCH):
            batches.append(cur)
            cur, longest = [], 0
        cur.append(it)
        longest = max(longest, n)
    return batches + ([cur] if cur else [])

def make_optimizer(model, total):
    groups = [{"params": model.lora_params(), "lr": LR},
              {"params": model.head.parameters(), "lr": HEAD_LR}]
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    warm = max(1, int(0.06 * total))
    factor = lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm)))
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, factor)

def with_distractors(it, aug):
    if it["task"] != "routing":
        return dict(it)
    lo, hi = (10, AUG_MAX_AGENTS) if it.get("many_agents") else (len(it["options"]), 9)
    return add_distractors(it, aug, aug.randint(lo, hi))

def prepare_batch(batch, aug):
    items = [with_distractors(it, aug) for it in batch]   # a new set of extra agents every time
    for it in items:                                      # so position can't be learned
        names = list(it["options"])
        random.shuffle(names)
        it["options"] = {n: it["options"][n] for n in names}
    return items

One fp16 training step, the progress line, the dev check, and the collapse guard of `03`.

In [9]:
def train_step(model, items, opt, sched, scaler, params):
    target = torch.tensor([list(it["options"]).index(it["answer"]) for it in items], device=DEVICE)
    loss = F.cross_entropy(model.scores(items), target)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(opt)
    scaler.update()
    sched.step()
    return loss.item()

def log_progress(epoch, step, total, run, t0):
    speed = step / (time.perf_counter() - t0)
    loss_r = statistics.mean(run["routing"][-50:] or [0])
    loss_q = statistics.mean(run["questions"][-50:] or [0])
    log(f"  epoch {epoch} step {step}/{total} | loss routing {loss_r:.4f} "
        f"questions {loss_q:.4f} | {speed:.2f} steps/s | "
        f"~{(total - step) / speed / 60:.1f} min left | {gpu()}")

def dev_accuracy(model, items, batch=8):
    model.train(False)
    ok = 0
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            picks = model.scores(chunk).argmax(1).tolist()
            ok += sum(list(it["options"])[k] == it["answer"] for it, k in zip(chunk, picks))
    return ok / max(1, len(items)), ok

def dev_check(model):
    acc_r, ok_r = dev_accuracy(model, dev_route)
    acc_m, ok_m = dev_accuracy(model, dev_route_many)
    acc_c, ok_c = dev_accuracy(model, dev_choice)
    msg = (f"dev routing {ok_r}/{len(dev_route)} ({acc_r:.1%}) | "
           f"with {AUG_MAX_AGENTS} agents {ok_m}/{len(dev_route_many)} ({acc_m:.1%}) | "
           f"dev questions {ok_c}/{len(dev_choice)} ({acc_c:.1%})")
    return acc_r, acc_c, msg

class Collapsed(Exception):
    """Training collapsed (dev too low after epoch 1); retry with another seed."""

COLLAPSE_DEV = {"routing": 0.95, "questions": 0.85}

The training loop; the best epoch's weights are restored at the end.

In [10]:
def snapshot(model):
    lora = {n: p.detach().clone() for n, p in model.enc.named_parameters() if p.requires_grad}
    return {"lora": lora, "head": copy.deepcopy(model.head.state_dict())}

def train_model(model, seed):
    batches = make_batches(train_route) + make_batches(train_choice)
    params = model.lora_params() + list(model.head.parameters())
    total = EPOCHS * len(batches)
    opt, sched = make_optimizer(model, total)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best_acc, best, best_epoch = -1.0, None, 0
    step, t0 = 0, time.perf_counter()
    log(f"training: {len(batches)} batches per epoch "
        f"({len(train_route)} routing + {len(train_choice)} questions), {total} steps")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        order = list(range(len(batches)))
        random.Random(seed + epoch).shuffle(order)
        aug = random.Random(seed * 7 + epoch)
        run = {"routing": [], "questions": []}
        for bi in order:
            items = prepare_batch(batches[bi], aug)
            loss = train_step(model, items, opt, sched, scaler, params)
            run["routing" if items[0]["task"] == "routing" else "questions"].append(loss)
            step += 1
            if step % 100 == 0 or step == total:
                log_progress(epoch, step, total, run, t0)
        acc_r, acc_c, msg = dev_check(model)
        log(f"epoch {epoch} done | {msg}")
        if epoch == 1 and (acc_r < COLLAPSE_DEV["routing"] or acc_c < COLLAPSE_DEV["questions"]):
            raise Collapsed(f"{msg} after epoch 1 (thresholds {COLLAPSE_DEV})")
        if (acc_r + acc_c) / 2 > best_acc:
            best_acc, best, best_epoch = (acc_r + acc_c) / 2, snapshot(model), epoch
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best["lora"].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best["head"])
    minutes = (time.perf_counter() - t0) / 60
    log(f"training done in {minutes:.1f} min | best epoch {best_epoch} "
        f"(mean dev accuracy {best_acc:.1%})")
    return {"train_minutes": minutes, "best_epoch": best_epoch}

Test helpers: the banking agents of `02` and `03`, GPU-synchronized timing, and a right / total table.

In [11]:
BANKING_AGENTS = {                  # exactly as in 02_frozen_router.ipynb, Cell 9, and 03
    "Payments Agent": ("Handles bank transfers, failed or pending payments, transaction status, "
        "payment charges and duplicate charges, refunds for transactions, and settlement problems "
        "where money was sent but not received. Does not handle card servicing such as blocking, "
        "replacing or activating cards."),
    "Card Support Agent": ("Handles debit and credit card servicing: card activation, blocking "
        "and unblocking, lost or stolen cards, card replacement and delivery, card limits, card "
        "PIN, card security and compromised card details, and card-specific transactions."),
    "Investment Agent": ("Handles mutual funds, SIPs, investments, portfolio allocation, "
        "investment transactions, redemptions and investment planning."),
    "Insurance Agent": ("Handles insurance policies, premiums, renewals, claims, policy expiry "
        "and policy servicing."),
    "General Customer Support Agent": ("Handles general account questions, profile and contact "
        "details, issues that do not clearly belong to a specialist, and routing customers to the "
        "right agent."),
}

def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "n": len(ms)}

def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000

def table(rows, key, cols, groups=None):
    """Print right / total per group (key(row) -> group) and for ALL rows."""
    groups = list(groups or sorted({key(r) for r in rows})) + ["ALL"]
    w = max(len(str(g)) for g in groups)
    print(f"  {'':{w}s} | {'n':>4} | " + " | ".join(f"{c:>12s}" for c in cols))
    for g in groups:
        rs = [r for r in rows if g == "ALL" or key(r) == g]
        if rs:
            counts = [f"{sum(map(is_right, rs)):>5} / {len(rs):<4}" for is_right in cols.values()]
            print(f"  {g:{w}s} | {len(rs):>4} | " + " | ".join(counts))

Counting helpers, live routing (the model's own previous pick as current agent, or with `given` the correct one) and first messages (top-1, top-2).

In [12]:
TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}
right = {"right": lambda r: r["ok"]}
SIZES = ["2-5", "6-10", "11-15", "16-30", "31-50"]

def n_ok(rows, key="ok"):
    return sum(r[key] for r in rows), len(rows)

def size(n):
    return ("2-5" if n <= 5 else "6-10" if n <= 10 else "11-15" if n <= 15
            else "16-30" if n <= 30 else "31-50")

def user(text):
    return [{"role": "user", "content": text}]

def live_routing(model, convs, given=False):
    rows, lat = [], []
    for conv in convs:
        cur = None
        for u in conv["user_turns"]:
            item = {"messages": conv["messages"][:u["message_index"] + 1], "current": cur,
                    "question": ROUTE_Q, "options": conv["agents"]}
            p, ms = timed_probs(model, item)
            pick = list(conv["agents"])[int(p.argmax())]
            rows.append({"company": conv.get("company"), "type": u["turn_type"],
                         "size": size(len(conv["agents"])), "ok": pick == u["expected_agent"]})
            lat.append(ms)
            cur = u["expected_agent"] if given else pick
    return rows, lat

def first_messages(model, msgs, reverse=False):
    rows, lat = [], []
    for m in msgs:
        names = list(m["agents"])[::-1] if reverse else list(m["agents"])
        item = {"messages": user(m["message"]), "current": None, "question": ROUTE_Q,
                "options": {n: m["agents"][n] for n in names}}
        p, ms = timed_probs(model, item)
        top = p.topk(2).indices.tolist()
        rows.append({"type": m["type"], "size": size(len(names)), "pick": names[top[0]],
                     "ok": names[top[0]] == m["target_agent"],
                     "top2": m["target_agent"] in [names[i] for i in top]})
        lat.append(ms)
    return rows, lat

Questions, then the routing and first-message tests, including the option-order check (the agents reversed).

In [13]:
def questions(model, items):
    rows, lat = [], []
    for it in items:
        item = {"messages": user(it["state"]), "current": None, "question": it["question"],
                "options": {o: "" for o in it["options"]}}
        p, ms = timed_probs(model, item)
        top = int(p.argmax())
        level_error = abs(top - it["answer_index"]) if it["type"] == "score" else None
        rows.append({**it, "size": size(len(it["options"])), "pick": it["options"][top],
                     "ok": it["options"][top] == it["answer"], "conf": p.max().item(),
                     "level_error": level_error})
        lat.append(ms)
    return rows, lat

def test_routing(model, R, LAT):
    for name, convs in [("E10", real_convs), ("IND", ind_r), ("IND2", ind2_r), ("MANY", many_r)]:
        log(f"{name}: routing every turn")
        rows, LAT[f"{name} routing"] = live_routing(model, convs)
        given_rows, _ = live_routing(model, convs, given=True)
        R[f"{name} routing live"] = n_ok(rows)
        R[f"{name} routing given"] = n_ok(given_rows)
        if name == "E10":
            R["E10 test companies"] = n_ok([r for r in rows if r["company"] in TEST_COMPANIES])
        print(f"{name} routing live (given previous agent: {R[f'{name} routing given'][0]})")
        table(rows, lambda r: r["type"], right)
        if name in ("IND2", "MANY"):
            table(rows, lambda r: r["size"] + " agents", right, [s + " agents" for s in SIZES])

def test_first_messages(model, R, LAT):
    for name, msgs in [("E11", e11), ("IND", ind_f), ("IND2", ind2_f), ("MANY", many_f)]:
        log(f"{name}: first messages")
        rows, LAT[f"{name} first"] = first_messages(model, msgs)
        R[f"{name} first top-1"] = n_ok(rows)
        R[f"{name} first top-2"] = n_ok(rows, "top2")
        table(rows, lambda r: r["type"], {"top-1": lambda r: r["ok"], "top-2": lambda r: r["top2"]})
        if name == "MANY":
            rev, _ = first_messages(model, msgs, reverse=True)
            same = sum(a["pick"] == b["pick"] for a, b in zip(rows, rev))
            R["option order: same pick reversed"] = (same, len(rows))
            print(f"  option order: same pick with the agents reversed {same} / {len(rows)} | "
                  f"top-1 forward {R['MANY first top-1'][0]}, reversed {n_ok(rev)[0]}")

The question tests (E12 lists its mistakes; the 22 flagged claim checks are also left out once) and `evaluate`, which runs everything, banking included.

In [14]:
def test_questions(model, R, LAT):
    for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q), ("MANY", many_q)]:
        log(f"{name}: questions")
        rows, LAT[f"{name} questions"] = questions(model, items)
        R[f"{name} questions"] = n_ok(rows)
        if name != "MANY":
            for t in TYPE:
                if any(r["type"] == t for r in rows):
                    R[f"{name} {TYPE[t]}"] = n_ok([r for r in rows if r["type"] == t])
            table(rows, lambda r: TYPE[r["type"]], right, [TYPE[t] for t in TYPE])
            sc = [r for r in rows if r["type"] == "score"]
            print(f"  score within 1 level: {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)}")
        else:
            table(rows, lambda r: r["family"], right)
            table(rows, lambda r: r["size"] + " options", right, [s + " options" for s in SIZES])
        if name == "E12":
            print("  mistakes:")
            for r in [r for r in rows if not r["ok"]]:
                print(f"    {r['id']} {r['type']:6s} | {r['state'][:44]:44s} | "
                      f"{r['question'][:36]:36s} | expected {r['answer']:14s} | "
                      f"got {r['pick']} ({r['conf']:.2f})")
        if name == "IND":
            R["IND questions unflagged"] = n_ok([r for r in rows if not r.get("flag")])

def evaluate(model):
    model.train(False)
    timed_probs(model, {"messages": user("warm-up"), "current": None, "question": ROUTE_Q,
                        "options": BANKING_AGENTS})
    R, LAT = {}, {}
    test_routing(model, R, LAT)
    test_first_messages(model, R, LAT)
    log("banking")
    rows, LAT["banking"] = [], []
    for ex in banking:
        p, ms = timed_probs(model, {"messages": ex["messages"], "current": ex["current_agent"],
                                    "question": ROUTE_Q, "options": BANKING_AGENTS})
        rows.append({"ok": list(BANKING_AGENTS)[int(p.argmax())] == ex["target_agent"]})
        LAT["banking"].append(ms)
    R["banking 50"], R["banking 225"] = n_ok(rows[:50]), n_ok(rows)
    test_questions(model, R, LAT)
    for name, L in LAT.items():
        L = lat_summary(L)
        print(f"latency {name:16s} p50 {L['p50']:4.0f} ms | p95 {L['p95']:4.0f} ms "
              f"({L['n']} questions)")
    return R, LAT

Train (a collapse restarts with the next seed), then run every test and write `one_pass_results.json`. **Expect** about 40 minutes on a T4 and run 28's results within about ±4.

In [15]:
attempts = []
for attempt in range(3):
    seed = SEED + 1000 * attempt
    random.seed(seed)
    torch.manual_seed(seed)
    if AMP:
        torch.cuda.reset_peak_memory_stats(DEVICE)
    model = OnePassModel()
    head_m = sum(p.numel() for p in model.head.parameters()) / 1e6
    log(f"attempt {attempt + 1} (seed {seed}) | head {head_m:.2f} M | {gpu()}")
    model.enc.print_trainable_parameters()
    try:
        trained = train_model(model, seed)
        attempts.append({"seed": seed, "status": "ok"})
        break
    except Collapsed as e:
        log(f"attempt {attempt + 1} COLLAPSED: {e} -> restarting with a new seed")
        attempts.append({"seed": seed, "status": f"collapsed: {e}"})
        model = None                              # free the failed attempt
        gc.collect()
        if AMP:
            torch.cuda.empty_cache()
else:
    raise RuntimeError(f"training collapsed in all attempts: {attempts}")

R, LAT = evaluate(model)
peak_gb = torch.cuda.max_memory_allocated(DEVICE) / 1e9 if AMP else 0
summary = {"results": R, "latency": {k: lat_summary(v) for k, v in LAT.items()},
           "attempts": attempts, **trained, "peak_gpu_gb": peak_gb}
json.dump(summary, open("one_pass_results.json", "w"), indent=1)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.58GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[06:55:10] attempt 1 (seed 42) | head 0.26 M | GPU 1.6 GB (peak 1.6)
trainable params: 7,196,672 || all params: 401,978,368 || trainable%: 1.7903
[06:55:10] training: 1184 batches per epoch (6319 routing + 11730 questions), 2368 steps
[06:55:20]   epoch 1 step 100/2368 | loss routing 1.6063 questions 1.0171 | 10.41 steps/s | ~3.6 min left | GPU 1.7 GB (peak 13.0)
[06:55:27]   epoch 1 step 200/2368 | loss routing 0.9815 questions 0.6046 | 12.10 steps/s | ~3.0 min left | GPU 1.7 GB (peak 13.0)
[06:55:35]   epoch 1 step 300/2368 | loss routing 0.4068 questions 0.5780 | 12.22 steps/s | ~2.8 min left | GPU 1.7 GB (peak 13.0)
[06:55:42]   epoch 1 step 400/2368 | loss routing 0.2465 questions 0.4381 | 12.46 steps/s | ~2.6 min left | GPU 1.7 GB (peak 13.0)
[06:55:50]   epoch 1 step 500/2368 | loss routing 0.2473 questions 0.4663 | 12.52 steps/s | ~2.5 min left | GPU 1.7 GB (peak 13.2)
[06:55:58]   epoch 1 step 600/2368 | loss routing 0.1473 questions 0.3346 | 12.60 steps/s | ~2.3 min left | GP

With `SAVE_MODEL = 1`, save the model to `signpost-one-pass/` and a zip (nothing is published), then reload it with its loader in fp16 and fp32: the scores must match.

In [16]:
OUT = Path("signpost-one-pass")
if SAVE_MODEL:
    shutil.rmtree(OUT, ignore_errors=True)
    model.enc.save_pretrained(OUT / "adapter")              # the LoRA adapter alone, before merging
    merged = model.enc.merge_and_unload()                   # W <- W + (alpha / r) B A, 112 layers
    merged.half().save_pretrained(OUT / "encoder", safe_serialization=True)
    model.tok.save_pretrained(OUT / "encoder")
    head_state = {k: v.float().cpu() for k, v in model.head.state_dict().items()}
    torch.save({"state_dict": head_state, "hidden": model.H}, OUT / "head.pt")
    cfg = {"name": "Signpost one-pass", "format": "one-pass", "base_model": ENCODER,
           "license_of_base": "Apache-2.0", "sep": SEP, "max_len": MAX_LEN,
           "max_options": MAX_OPTIONS, "route_question": ROUTE_Q, "lora_r": LORA_R,
           "lora_alpha": LORA_ALPHA, "lora": f"peft {peft.__version__}", "score_loss": "plain",
           "epochs": EPOCHS, "best_epoch": trained["best_epoch"], "seed_attempts": attempts,
           "data_branch": DATA_BRANCH, "results": {k: v[0] for k, v in R.items()}}
    json.dump(cfg, open(OUT / "signpost_config.json", "w"), indent=1)
    shutil.copy(DATA_DIR.parent.parent / "signpost" / "signpost_one_pass.py",
                OUT / "signpost_one_pass.py")
    shutil.make_archive("signpost-one-pass", "zip", root_dir=".", base_dir=str(OUT))
    size_mb = {d: sum(f.stat().st_size for f in (OUT / d).rglob("*") if f.is_file()) / 1e6
               for d in ("encoder", "adapter")}
    zip_gb = Path("signpost-one-pass.zip").stat().st_size / 1e9
    log(f"saved {OUT}/ (encoder {size_mb['encoder']:.0f} MB, adapter {size_mb['adapter']:.0f} MB) "
        f"and signpost-one-pass.zip ({zip_gb:.2f} GB)")

    sys.path.insert(0, str(OUT.resolve()))
    from signpost_one_pass import SignpostOnePass
    for dtype in (torch.float16, torch.float32):            # fp16 = GPU use; fp32 = a CPU run
        sp = SignpostOnePass(OUT, device=DEVICE, dtype=dtype)
        n11 = sum(sp.route(user(m["message"]), m["agents"])[0] == m["target_agent"] for m in e11)
        n12 = sum(sp.choice(it["state"], it["question"], it["options"])[0] == it["answer"]
                  for it in e12)
        nmany = sum(sp.choice(it["state"], it["question"], it["options"])[0] == it["answer"]
                    for it in many_q)
        log(f"reload check ({dtype}): E11 {n11}/{len(e11)} "
            f"(trained model {R['E11 first top-1'][0]}) | "
            f"E12 {n12}/{len(e12)} (trained model {R['E12 questions'][0]}) | "
            f"many-option {nmany}/{len(many_q)} (trained model {R['MANY questions'][0]})")
        del sp

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[06:59:29] saved signpost-one-pass/ (encoder 793 MB, adapter 29 MB) and signpost-one-pass.zip (0.76 GB)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[06:59:32] reload check (torch.float16): E11 98/100 (trained model 98) | E12 157/170 (trained model 157) | many-option 151/162 (trained model 151)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[06:59:36] reload check (torch.float32): E11 98/100 (trained model 98) | E12 157/170 (trained model 157) | many-option 151/162 (trained model 151)


This run next to measured results on the same items: run 28, the published Signpost and the 2B decider (`04`). Each training run lands a little differently: the same recipe and seed differ by about ±4 items on most tests, and more on the independent questions, so compare the columns within that range.

In [17]:
REF = {
    "run 28": {"E10 routing live": 179, "E10 test companies": 76, "IND routing live": 134,
               "IND2 routing live": 55, "MANY routing live": 100, "E11 first top-1": 97,
               "IND first top-1": 180, "IND2 first top-1": 49, "MANY first top-1": 63,
               "banking 50": 49, "banking 225": 212, "E12 questions": 158, "E12 yes/no": 57,
               "E12 choice": 56, "E12 score": 45, "IND questions": 221, "IND yes/no": 70,
               "IND choice": 105, "IND score": 46, "IND2 questions": 185, "IND2 yes/no": 44,
               "IND2 choice": 47, "IND2 score": 46, "IND2 many-option": 48, "MANY questions": 147,
               "option order: same pick reversed": 68},
    "Signpost (published)": {"E10 routing live": 181, "E10 test companies": 78,
                             "IND routing live": 133, "IND2 routing live": 62,
                             "IND2 routing given": 65, "MANY routing live": 103,
                             "MANY routing given": 109, "E11 first top-1": 97,
                             "IND first top-1": 180, "IND2 first top-1": 47, "MANY first top-1": 62,
                             "banking 50": 48, "banking 225": 210, "E12 questions": 161,
                             "E12 yes/no": 58, "E12 choice": 58, "E12 score": 45,
                             "IND questions": 221, "IND yes/no": 75, "IND choice": 103,
                             "IND score": 43, "IND2 questions": 188, "IND2 yes/no": 47,
                             "IND2 choice": 47, "IND2 score": 45, "IND2 many-option": 49,
                             "MANY questions": 150},
    "decider 2B": {"E10 routing live": 151, "E10 test companies": 62, "IND routing live": 113,
                   "IND2 routing live": 55, "MANY routing live": 83, "E11 first top-1": 96,
                   "E11 first top-2": 100, "IND first top-1": 178, "IND first top-2": 193,
                   "IND2 first top-1": 50, "MANY first top-1": 70, "banking 50": 48,
                   "banking 225": 214, "E12 questions": 160, "E12 yes/no": 56, "E12 choice": 56,
                   "E12 score": 48, "IND questions": 253, "IND yes/no": 77, "IND choice": 106,
                   "IND score": 70, "IND questions unflagged": 253, "IND2 questions": 185,
                   "IND2 yes/no": 49, "IND2 choice": 46, "IND2 score": 43, "IND2 many-option": 47,
                   "MANY questions": 160},
}

print(f"{'result':34s} | {'this run':>11} | " + " | ".join(f"{c:>20s}" for c in REF))
for k, (ok_, n) in R.items():
    refs = " | ".join(f"{REF[c].get(k, '—'):>20}" for c in REF)
    print(f"{k:34s} | {f'{ok_} / {n}':>11} | " + refs)

result                             |    this run |               run 28 | Signpost (published) |           decider 2B
E10 routing live                   |   177 / 193 |                  179 |                  181 |                  151
E10 routing given                  |   180 / 193 |                    — |                    — |                    —
E10 test companies                 |     74 / 83 |                   76 |                   78 |                   62
IND routing live                   |   128 / 172 |                  134 |                  133 |                  113
IND routing given                  |   136 / 172 |                    — |                    — |                    —
IND2 routing live                  |     57 / 70 |                   55 |                   62 |                   55
IND2 routing given                 |     60 / 70 |                    — |                   65 |                    —
MANY routing live                  |   103 / 120 |      

Latency p50 in ms on a T4, then the training summary.

In [18]:
P50 = {
    "Signpost": {"E10 routing": 58, "IND routing": 70, "IND2 routing": 187, "MANY routing": 99,
                 "E11 first": 57, "IND first": 57, "IND2 first": 73, "MANY first": 51,
                 "banking": 73, "E12 questions": 55, "IND questions": 55, "IND2 questions": 57,
                 "MANY questions": 63},
    "decider": {"E10 routing": 316, "IND routing": 376, "IND2 routing": 654, "MANY routing": 574,
                "E11 first": 297, "IND first": 287, "IND2 first": 463, "MANY first": 465,
                "banking": 468, "E12 questions": 197, "IND questions": 199, "IND2 questions": 200,
                "MANY questions": 382},
}

print(f"\n{'latency p50 (ms)':18s} | {'this run':>8} | {'Signpost':>8} | {'decider':>8}")
for k, v in LAT.items():
    this_run = lat_summary(v)["p50"]
    print(f"{k:18s} | {this_run:>8.0f} | {P50['Signpost'].get(k, '—'):>8} | "
          f"{P50['decider'].get(k, '—'):>8}")
print(f"\ntraining {trained['train_minutes']:.1f} min, best epoch {trained['best_epoch']} | "
      f"attempts {[a['status'] for a in attempts]} | peak GPU {peak_gb:.1f} GB")


latency p50 (ms)   | this run | Signpost |  decider
E10 routing        |       15 |       58 |      316
IND routing        |       15 |       70 |      376
IND2 routing       |       16 |      187 |      654
MANY routing       |       16 |       99 |      574
E11 first          |       15 |       57 |      297
IND first          |       15 |       57 |      287
IND2 first         |       15 |       73 |      463
MANY first         |       15 |       51 |      465
banking            |       16 |       73 |      468
E12 questions      |       14 |       55 |      197
IND questions      |       14 |       55 |      199
IND2 questions     |       14 |       57 |      200
MANY questions     |       15 |       63 |      382

training 3.3 min, best epoch 2 | attempts ['ok'] | peak GPU 13.3 GB


## What we tried and dropped

Correct / total on the same items; one seed per run, noise about ±4 (`experiments/modernbert_router_ledger.md`).

| Run | What we tried | Result | Decision |
|---|---|---|---|
| 26 | One pass per question, with **30% of Signpost's routing and question data** + the 2,500 many-option questions | Level with Signpost on most tests at a flat ~51–55 ms for 10–50 options; fresh tests: many-option 151 / 162 (Signpost 150, decider 160), many-agent routing 104 / 120 (103, decider 83). But E10 all −9 and E12 −7 | More data |
| 27 | **All of Signpost's data** + many-option questions + 2,500 routing items trained with 10–15 agents; ordinal score loss | Routing fixed (E10 all 177, banking 211, independent routing 136), but independent score fell to 35 / 92 and E12 to 153 | The ordinal loss hurt score here |
| 28 | The same with **plain cross-entropy** for score questions | Independent score 46 (questions 221 = Signpost), E12 158; level with Signpost on every old test (within ±3) at ~55–60 ms | **Kept: this notebook's recipe** |
| 29 | **No score questions** in training or tests | Nothing improved; E12 without score 113 → 107 and independent routing 134 → 128 | Dropped: score questions help the other jobs |

**One pass buys** flat latency (about 55 ms for 50 options; Signpost about 190 ms with 8–11 agents), **not** accuracy beyond Signpost's.

## Next

Load the saved model with `signpost/signpost_one_pass.py` (`route`, `yes_no`, `choice`, `score`, as in `signpost.py`).